# Sidebar Link Validator

This notebook ensures all sidebar links are properly linked to pages in the `/templates` directory. It will:
1. Check for existing links in the sidebar configuration
2. Validate if corresponding pages exist
3. Create missing pages as needed
4. Update sidebar links if necessary
5. Generate a comprehensive link status report

## Import Required Libraries

In [ ]:
# Import necessary libraries
import os
import json
import datetime
import re
import shutil
from pathlib import Path

## Load Sidebar Links
Parse the sidebar configuration file to extract all defined links.

In [ ]:
def load_sidebar_config(config_path="templates/sidebar_config.json"):
    """
    Load the sidebar configuration from the JSON file.
    
    Args:
        config_path (str): Path to the sidebar configuration file
        
    Returns:
        dict: The sidebar configuration as a dictionary
    """
    try:
        with open(config_path, 'r') as f:
            sidebar_config = json.load(f)
        print(f"Successfully loaded sidebar configuration from {config_path}")
        return sidebar_config
    except FileNotFoundError:
        print(f"Sidebar configuration file not found at {config_path}")
        return None
    except json.JSONDecodeError:
        print(f"Error parsing sidebar configuration file at {config_path}")
        return None

# Load the sidebar configuration
sidebar_config = load_sidebar_config()

# Function to extract all links from the sidebar configuration
def extract_links(config):
    """
    Extract all links from the sidebar configuration.
    
    Args:
        config (dict): The sidebar configuration
        
    Returns:
        list: A list of all links in the sidebar
    """
    if not config:
        return []
    
    links = []
    
    def process_item(item):
        if "link" in item:
            links.append(item["link"])
        if "children" in item:
            for child in item["children"]:
                process_item(child)
    
    for section in config:
        process_item(section)
    
    return links

# Extract all links from the sidebar configuration
if sidebar_config:
    all_links = extract_links(sidebar_config)
    print(f"Found {len(all_links)} links in the sidebar configuration:")
    for link in all_links:
        print(f"  - {link}")
else:
    all_links = []

## Check Link Validity
Iterate through the extracted links and check if the corresponding pages exist in the `/templates` directory.

In [ ]:
def check_link_validity(links, templates_dir="templates"):
    """
    Check if the pages corresponding to the links exist in the templates directory.
    
    Args:
        links (list): A list of links to check
        templates_dir (str): Path to the templates directory
        
    Returns:
        tuple: Lists of valid and missing links
    """
    valid_links = []
    missing_links = []
    
    for link in links:
        # Normalize the link (remove leading slash if present)
        normalized_link = link.lstrip('/')
        
        # Handle index or home page
        if normalized_link == "" or normalized_link == "index" or normalized_link == "home":
            template_path = os.path.join(templates_dir, "index.html")
        # Handle other pages
        else:
            # Check if the link ends with .html
            if normalized_link.endswith('.html'):
                template_path = os.path.join(templates_dir, normalized_link)
            else:
                # Try both with and without .html extension
                template_path = os.path.join(templates_dir, f"{normalized_link}.html")
                template_path_alt = os.path.join(templates_dir, normalized_link, "index.html")
                
                # If neither exists, use the .html version as the missing link
                if not os.path.exists(template_path) and not os.path.exists(template_path_alt):
                    missing_links.append((link, template_path))
                    continue
                
                # If one exists, use that as the valid link
                if os.path.exists(template_path):
                    valid_links.append((link, template_path))
                    continue
                elif os.path.exists(template_path_alt):
                    valid_links.append((link, template_path_alt))
                    continue
        
        # Check if the template exists
        if os.path.exists(template_path):
            valid_links.append((link, template_path))
        else:
            missing_links.append((link, template_path))
    
    return valid_links, missing_links

# Check the validity of the links
valid_links, missing_links = check_link_validity(all_links)

print(f"\nValid links ({len(valid_links)}):")
for link, path in valid_links:
    print(f"  - {link} -> {path}")

print(f"\nMissing links ({len(missing_links)}):")
for link, path in missing_links:
    print(f"  - {link} -> {path}")

## Create Missing Pages
For links without corresponding pages, create placeholder files in the `/templates` directory.

In [ ]:
def create_placeholder_template(template_path, title, description=None):
    """
    Create a placeholder HTML template.
    
    Args:
        template_path (str): Path to create the template
        title (str): Title for the page
        description (str, optional): Description for the page
        
    Returns:
        bool: True if the template was created successfully, False otherwise
    """
    # Ensure the directory exists
    os.makedirs(os.path.dirname(template_path), exist_ok=True)
    
    # Create a basic HTML template
    title_clean = title.replace('-', ' ').replace('_', ' ').title()
    if description is None:
        description = f"Placeholder page for {title_clean}"
    
    template_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{title_clean}</title>
</head>
<body>
    <h1>{title_clean}</h1>
    <p>{description}</p>
    <p><em>This is a placeholder page created by the sidebar link validator on {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</em></p>
</body>
</html>
"""
    try:
        with open(template_path, 'w') as f:
            f.write(template_content)
        return True
    except Exception as e:
        print(f"Error creating template at {template_path}: {e}")
        return False

def create_missing_pages(missing_links):
    """
    Create placeholder pages for missing links.
    
    Args:
        missing_links (list): A list of tuples (link, path) for missing links
        
    Returns:
        list: A list of links that were successfully created
    """
    created_links = []
    
    for link, path in missing_links:
        # Extract title from the link or path
        title = os.path.splitext(os.path.basename(path))[0]
        if not title:
            title = os.path.basename(os.path.dirname(path))
        if not title:
            title = link.rstrip('/').split('/')[-1]
        if not title:
            title = "Untitled Page"
        
        # Create the placeholder template
        if create_placeholder_template(path, title):
            created_links.append((link, path))
    
    return created_links

# Create missing pages
created_links = create_missing_pages(missing_links)

print(f"\nCreated pages ({len(created_links)}):")
for link, path in created_links:
    print(f"  - {link} -> {path}")

## Update Sidebar Links
Ensure all sidebar links point to the correct files in the `/templates` directory.

In [ ]:
def normalize_link(link):
    """
    Normalize a link to ensure consistency.
    
    Args:
        link (str): The link to normalize
        
    Returns:
        str: The normalized link
    """
    # Ensure link starts with '/'
    if not link.startswith('/'):
        link = '/' + link
    
    # Remove trailing '/' if present
    if link != '/' and link.endswith('/'):
        link = link[:-1]
    
    return link

def update_sidebar_links(config, valid_links, created_links):
    """
    Update the sidebar links to point to the correct files.
    
    Args:
        config (dict): The sidebar configuration
        valid_links (list): A list of valid links
        created_links (list): A list of created links
        
    Returns:
        dict: The updated sidebar configuration
    """
    if not config:
        return config
    
    # Create a mapping of links to paths
    link_to_path = {}
    for link, path in valid_links + created_links:
        link_to_path[link] = path
    
    def process_item(item):
        if "link" in item:
            # Normalize the link
            item["link"] = normalize_link(item["link"])
        
        if "children" in item:
            for child in item["children"]:
                process_item(child)
    
    # Update all links in the configuration
    for section in config:
        process_item(section)
    
    return config

# Update sidebar links
if sidebar_config:
    updated_sidebar_config = update_sidebar_links(sidebar_config, valid_links, created_links)
    
    # Save the updated sidebar configuration
    updated_config_path = "templates/sidebar_config_updated.json"
    try:
        with open(updated_config_path, 'w') as f:
            json.dump(updated_sidebar_config, f, indent=2)
        print(f"\nUpdated sidebar configuration saved to {updated_config_path}")
    except Exception as e:
        print(f"\nError saving updated sidebar configuration: {e}")
else:
    print("\nNo sidebar configuration to update.")

## Generate Link Status Report
Generate a report listing which links are complete and working, and which were missing and created.

In [ ]:
def generate_link_status_report(valid_links, created_links, missing_links):
    """
    Generate a report on the status of all links.
    
    Args:
        valid_links (list): A list of valid links
        created_links (list): A list of created links
        missing_links (list): A list of missing links
        
    Returns:
        str: The report as a string
    """
    report = f"# Sidebar Link Status Report\n\n"
    report += f"Generated on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    
    # Summary
    report += f"## Summary\n\n"
    report += f"- Total links: {len(valid_links) + len(missing_links)}\n"
    report += f"- Valid links: {len(valid_links)}\n"
    report += f"- Created links: {len(created_links)}\n"
    report += f"- Still missing links: {len(missing_links) - len(created_links)}\n\n"
    
    # Valid links
    report += f"## Valid Links\n\n"
    if valid_links:
        for link, path in valid_links:
            report += f"- `{link}` -> `{path}`\n"
    else:
        report += "No valid links found.\n"
    report += "\n"
    
    # Created links
    report += f"## Created Links\n\n"
    if created_links:
        for link, path in created_links:
            report += f"- `{link}` -> `{path}`\n"
    else:
        report += "No links were created.\n"
    report += "\n"
    
    # Still missing links
    still_missing = [(link, path) for link, path in missing_links if (link, path) not in created_links]
    report += f"## Still Missing Links\n\n"
    if still_missing:
        for link, path in still_missing:
            report += f"- `{link}` -> `{path}`\n"
    else:
        report += "All missing links have been created.\n"
    
    return report

# Generate the link status report
report = generate_link_status_report(valid_links, created_links, missing_links)

# Save the report
report_path = "sidebar_link_report.md"
try:
    with open(report_path, 'w') as f:
        f.write(report)
    print(f"\nLink status report saved to {report_path}")
    print("\nReport content:")
    print(report)
except Exception as e:
    print(f"\nError saving link status report: {e}")

## Conclusion

This notebook has now:
1. Loaded and parsed the sidebar configuration
2. Identified valid and missing links
3. Created placeholder pages for missing links
4. Updated the sidebar configuration for consistency
5. Generated a comprehensive link status report

Next steps:
1. Review the created placeholder pages and enhance them with actual content
2. Verify the updated sidebar configuration and apply it to your application
3. Address any still-missing links
4. Run this notebook periodically to ensure continued link integrity

# Sidebar Link Validator

This notebook validates that all sidebar links have corresponding pages in the /templates directory, creating missing pages as needed.

## Import Required Libraries

Import libraries needed for file operations and data processing.

In [ ]:
import os
import json
import re
from pathlib import Path
import datetime

## Load Sidebar Links

Parse the sidebar configuration file to extract all links and their corresponding paths.

In [ ]:
# Define the path to the sidebar configuration file
# Assuming it's stored in a common location like static/config or similar
sidebar_config_path = "static/config/sidebar.json"

# Function to load and parse the sidebar configuration
def load_sidebar_config(config_path):
    try:
        with open(config_path, 'r') as f:
            sidebar_config = json.load(f)
        print(f"Successfully loaded sidebar configuration from {config_path}")
        return sidebar_config
    except FileNotFoundError:
        print(f"Error: Sidebar configuration file not found at {config_path}")
        return None
    except json.JSONDecodeError:
        print(f"Error: Invalid JSON in sidebar configuration file at {config_path}")
        return None

# Alternative function if the sidebar links are defined in a Python file
def extract_sidebar_links_from_py(py_file_path):
    sidebar_links = []
    try:
        with open(py_file_path, 'r') as f:
            content = f.read()
            
        # Use regex to find URL patterns - this is a simple example and may need adjustment
        # Looking for patterns like 'url': '/some/path' or url="/some/path"
        url_pattern = r'[\'"]url[\'"]\s*:\s*[\'"]([^\'"]+)[\'"]'
        matches = re.findall(url_pattern, content)
        
        for match in matches:
            sidebar_links.append(match)
            
        print(f"Extracted {len(sidebar_links)} links from {py_file_path}")
        return sidebar_links
    except Exception as e:
        print(f"Error extracting links from {py_file_path}: {str(e)}")
        return []

# Try to load the sidebar configuration
sidebar_config = None
if os.path.exists(sidebar_config_path):
    sidebar_config = load_sidebar_config(sidebar_config_path)
else:
    # If JSON config doesn't exist, try to find a Python file with sidebar definition
    possible_py_files = ["app.py", "routes.py", "navigation.py", "sidebar.py"]
    for py_file in possible_py_files:
        if os.path.exists(py_file):
            sidebar_links = extract_sidebar_links_from_py(py_file)
            if sidebar_links:
                print(f"Found sidebar links in {py_file}")
                break
    else:
        print("Could not find sidebar configuration in common locations.")
        # Prompt for manual path input
        custom_path = input("Enter the path to your sidebar configuration file: ")
        if custom_path.endswith('.json'):
            sidebar_config = load_sidebar_config(custom_path)
        elif custom_path.endswith('.py'):
            sidebar_links = extract_sidebar_links_from_py(custom_path)

## Extract Links from Sidebar Configuration

Process the loaded sidebar configuration to extract all unique links.

In [ ]:
def extract_links_from_config(config):
    """
    Recursively extract all links from the sidebar configuration.
    The structure might vary, but typically there are sections and subsections with links.
    """
    links = []
    
    # Function to recursively search for links in nested structures
    def search_links(item):
        if isinstance(item, dict):
            # If the item is a dictionary, look for 'url' or 'link' key
            if 'url' in item:
                links.append(item['url'])
            elif 'link' in item:
                links.append(item['link'])
            
            # Recursively search in all values
            for value in item.values():
                search_links(value)
        
        elif isinstance(item, list):
            # If the item is a list, search each element
            for element in item:
                search_links(element)
    
    # Start the recursive search
    search_links(config)
    
    # Clean the links - remove any query parameters or fragments
    cleaned_links = []
    for link in links:
        # Remove query parameters and fragments
        clean_link = link.split('?')[0].split('#')[0]
        # Ensure it starts with a slash for template paths
        if not clean_link.startswith('/'):
            clean_link = '/' + clean_link
        cleaned_links.append(clean_link)
    
    return cleaned_links

# Process the sidebar configuration to extract links
sidebar_links = []
if sidebar_config:
    sidebar_links = extract_links_from_config(sidebar_config)
    print(f"Extracted {len(sidebar_links)} unique links from the sidebar configuration")
    
# Display the extracted links
print("Extracted sidebar links:")
for link in sidebar_links:
    print(f" - {link}")

## Check for Missing Pages

Iterate through the extracted links and check if the corresponding files exist in the /templates directory.

In [ ]:
# Define the templates directory
templates_dir = "templates"

def check_missing_pages(links, templates_directory):
    """
    Check if template files exist for each link.
    Returns a tuple of (existing_links, missing_links)
    """
    existing_links = []
    missing_links = []
    
    # Make sure the templates directory exists
    if not os.path.exists(templates_directory):
        print(f"Warning: Templates directory '{templates_directory}' does not exist.")
        # Create the templates directory if it doesn't exist
        os.makedirs(templates_directory)
        print(f"Created templates directory: {templates_directory}")
    
    for link in links:
        # Convert the link to a template path
        # Remove leading slash if present
        if link.startswith('/'):
            link = link[1:]
        
        # If link is empty or just '/', it's likely the home page
        if not link or link == '/':
            template_path = os.path.join(templates_directory, "index.html")
        else:
            # Handle different possible template extensions
            template_path = os.path.join(templates_directory, f"{link}.html")
            # Also check for index.html in subdirectories
            if link.endswith('/'):
                alternative_path = os.path.join(templates_directory, link, "index.html")
            else:
                alternative_path = os.path.join(templates_directory, link, "index.html")
            
            # Check if the directory exists but not as a file
            if os.path.isdir(os.path.join(templates_directory, link)):
                template_path = os.path.join(templates_directory, link, "index.html")
        
        # Check if the file exists
        if os.path.exists(template_path):
            existing_links.append((link, template_path))
        elif 'alternative_path' in locals() and os.path.exists(alternative_path):
            existing_links.append((link, alternative_path))
        else:
            missing_links.append(link)
    
    return existing_links, missing_links

# Check for missing pages
existing_links, missing_links = check_missing_pages(sidebar_links, templates_dir)

# Display the results
print(f"\nFound {len(existing_links)} existing pages:")
for link, path in existing_links:
    print(f" - {link} -> {path}")

print(f"\nFound {len(missing_links)} missing pages:")
for link in missing_links:
    print(f" - {link}")

## Create Missing Pages

For each missing page, create a new file in the /templates directory with a basic template structure.

In [ ]:
def create_template_page(link, templates_directory):
    """
    Create a new template page for the given link.
    """
    # Remove leading slash if present
    if link.startswith('/'):
        link = link[1:]
    
    # If link is empty or just '/', it's the home page
    if not link or link == '/':
        template_path = os.path.join(templates_directory, "index.html")
        template_title = "Home"
    else:
        # Convert link to template path
        template_path = os.path.join(templates_directory, f"{link}.html")
        # Create directory if needed
        os.makedirs(os.path.dirname(template_path), exist_ok=True)
        # Generate a title from the link
        template_title = link.replace('/', ' ').replace('-', ' ').replace('_', ' ').title()
    
    # Basic template content
    template_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>{template_title}</title>
    <link rel="stylesheet" href="/static/css/style.css">
</head>
<body>
    <header>
        <h1>{template_title}</h1>
    </header>
    
    <main>
        <p>This is the {template_title} page.</p>
        <p>Created automatically by the sidebar link validator on {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}</p>
    </main>
    
    <footer>
        <p>&copy; {datetime.datetime.now().year} Your Company Name</p>
    </footer>
    
    <script src="/static/js/script.js"></script>
</body>
</html>
"""
    
    # Write the template file
    try:
        with open(template_path, 'w') as f:
            f.write(template_content)
        print(f"Created template: {template_path}")
        return template_path
    except Exception as e:
        print(f"Error creating template {template_path}: {str(e)}")
        return None

# Create missing pages
created_pages = []
for link in missing_links:
    template_path = create_template_page(link, templates_dir)
    if template_path:
        created_pages.append((link, template_path))

# Display the results
print(f"\nCreated {len(created_pages)} new template pages:")
for link, path in created_pages:
    print(f" - {link} -> {path}")

## Update Sidebar Links

Check if any sidebar links need to be updated to point to the correct paths in the /templates directory.

In [ ]:
def update_sidebar_links(sidebar_config, existing_links, created_pages):
    """
    Update the sidebar configuration to ensure all links point to existing pages.
    This is a placeholder function that would need to be customized based on the
    actual structure of your sidebar configuration.
    """
    # This is a simplified example - you would need to adapt this to your actual sidebar structure
    updated = False
    
    def update_links_in_config(config):
        nonlocal updated
        if isinstance(config, dict):
            if 'url' in config:
                link = config['url']
                # Check if this link was in the missing links and has been created
                for missing_link, created_path in created_pages:
                    if link == missing_link:
                        # The link might need to be updated based on the created path
                        # This depends on your routing system
                        print(f"Link '{link}' now has a corresponding template at {created_path}")
                        updated = True
            
            # Recursively update all values
            for key, value in config.items():
                update_links_in_config(value)
        
        elif isinstance(config, list):
            for item in config:
                update_links_in_config(item)
    
    # Start the recursive update
    if sidebar_config:
        update_links_in_config(sidebar_config)
    
    return updated

# Update the sidebar links if needed
if sidebar_config and created_pages:
    updated = update_sidebar_links(sidebar_config, existing_links, created_pages)
    if updated:
        print("\nSidebar links have been updated to point to the newly created pages")
        
        # If desired, save the updated sidebar configuration back to the file
        # with open(sidebar_config_path, 'w') as f:
        #     json.dump(sidebar_config, f, indent=2)
        #     print(f"Updated sidebar configuration saved to {sidebar_config_path}")
    else:
        print("\nNo updates to sidebar links were necessary")

## Generate Report of Link Status

Create a report listing which links are complete and working, and which links were missing and have been created.

In [ ]:
def generate_link_status_report(existing_links, created_pages):
    """
    Generate a report of the link status.
    """
    report = {
        "timestamp": datetime.datetime.now().isoformat(),
        "total_links": len(existing_links) + len(created_pages),
        "existing_links": [{"link": link, "template": path} for link, path in existing_links],
        "created_links": [{"link": link, "template": path} for link, path in created_pages]
    }
    
    # Print a simple summary
    print("\n=== Sidebar Link Status Report ===")
    print(f"Generated: {report['timestamp']}")
    print(f"Total links processed: {report['total_links']}")
    print(f"Existing links: {len(existing_links)}")
    print(f"Created links: {len(created_pages)}")
    
    # Generate a detailed table-like output
    print("\nDetailed Status:")
    print("| Status  | Link | Template Path |")
    print("|---------|------|---------------|")
    for link, path in existing_links:
        print(f"| Existing | {link} | {path} |")
    for link, path in created_pages:
        print(f"| Created | {link} | {path} |")
    
    # Save the report to a JSON file
    report_path = "link_status_report.json"
    try:
        with open(report_path, 'w') as f:
            json.dump(report, f, indent=2)
        print(f"\nDetailed report saved to {report_path}")
    except Exception as e:
        print(f"\nError saving report to {report_path}: {str(e)}")
    
    return report

# Generate the link status report
link_status_report = generate_link_status_report(existing_links, created_pages)

## Conclusion

All sidebar links have been checked and missing pages have been created. The link status report provides a summary of the process and results.

Next steps:
1. Review the newly created template pages and customize them as needed
2. Ensure that the sidebar configuration correctly points to all template pages
3. Run this notebook periodically to catch any new links added to the sidebar

# Sidebar Links Validator

This notebook will:
1. Parse the sidebar configuration to extract all links
2. Check if corresponding template files exist in the /templates directory
3. Create missing template files with placeholder content
4. Update sidebar links to ensure they point to valid files
5. Generate a status report for all links

## Import Required Libraries

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
import datetime
import shutil
import re

# For nice display of results
from IPython.display import display, HTML, Markdown

## Set Project Paths and Constants

In [ ]:
# Define the base directories
BASE_DIR = Path('.')
TEMPLATES_DIR = BASE_DIR / 'templates'
CONFIG_DIR = BASE_DIR / 'config'

# Sidebar configuration file path
SIDEBAR_CONFIG_PATH = CONFIG_DIR / 'sidebar_config.json'

# Template for new pages
DEFAULT_TEMPLATE = """<!-- Page: {title} -->
<h1>{title}</h1>
<p>This is an auto-generated placeholder page for {title}.</p>
<p>Created on: {date}</p>
"""

# Check if directories exist
print(f"Templates directory exists: {TEMPLATES_DIR.exists()}")
print(f"Config directory exists: {CONFIG_DIR.exists()}")
print(f"Sidebar config exists: {SIDEBAR_CONFIG_PATH.exists()}")

## Load Sidebar Links

Parse the sidebar configuration file to extract all links and their details.

In [ ]:
def load_sidebar_config():
    """Load and parse the sidebar configuration file."""
    if not SIDEBAR_CONFIG_PATH.exists():
        print("⚠️ Sidebar configuration file not found.")
        return {}
    
    try:
        with open(SIDEBAR_CONFIG_PATH, 'r') as f:
            sidebar_config = json.load(f)
        print(f"✅ Successfully loaded sidebar configuration.")
        return sidebar_config
    except json.JSONDecodeError:
        print("❌ Error parsing sidebar configuration file. Invalid JSON.")
        return {}
    except Exception as e:
        print(f"❌ Error loading sidebar configuration: {str(e)}")
        return {}

def extract_links(sidebar_config):
    """Extract all links from the sidebar configuration."""
    links = []
    
    def process_item(item, parent_path=""):
        if 'link' in item:
            link_path = item['link']
            links.append({
                'title': item.get('title', 'Untitled'),
                'link': link_path,
                'parent_path': parent_path
            })
        
        # Process child items if they exist
        if 'children' in item:
            current_path = parent_path + "/" + item.get('title', '') if parent_path else item.get('title', '')
            for child in item['children']:
                process_item(child, current_path)
    
    # Process each top-level item in the sidebar
    for item in sidebar_config.get('sidebar', []):
        process_item(item)
    
    return links

# Load the sidebar configuration and extract links
sidebar_config = load_sidebar_config()
sidebar_links = extract_links(sidebar_config)

# Display the links
if sidebar_links:
    print(f"Found {len(sidebar_links)} links in the sidebar configuration:")
    links_df = pd.DataFrame(sidebar_links)
    display(links_df)
else:
    print("No sidebar links found.")

## Check for Missing Pages

Iterate through the extracted links and check if the corresponding template files exist.

In [ ]:
def normalize_link_to_filepath(link):
    """Convert a link path to a file system path."""
    # Remove leading slash if present
    if link.startswith('/'):
        link = link[1:]
    
    # Handle special cases (index, etc.)
    if link == '' or link == '/':
        link = 'index.html'
    
    # Add .html extension if not present
    if not link.endswith('.html'):
        link = f"{link}.html"
    
    return link

def check_template_exists(link):
    """Check if a template file exists for the given link."""
    file_path = normalize_link_to_filepath(link)
    template_path = TEMPLATES_DIR / file_path
    
    return {
        'link': link,
        'file_path': str(file_path),
        'template_path': str(template_path),
        'exists': template_path.exists()
    }

# Check each link for corresponding template file
template_status = []
for link_info in sidebar_links:
    link = link_info['link']
    result = check_template_exists(link)
    result['title'] = link_info['title']
    result['parent_path'] = link_info['parent_path']
    template_status.append(result)

# Create a DataFrame for better visualization
status_df = pd.DataFrame(template_status)
status_df['status'] = status_df['exists'].apply(lambda x: '✅ Exists' if x else '❌ Missing')

# Display the results
print(f"\nTemplate Status Summary:")
print(f"Total Links: {len(status_df)}")
print(f"Existing Templates: {status_df['exists'].sum()}")
print(f"Missing Templates: {len(status_df) - status_df['exists'].sum()}")

display(status_df[['title', 'parent_path', 'link', 'file_path', 'status']])

# Get missing templates
missing_templates = status_df[~status_df['exists']]
if not missing_templates.empty:
    print(f"\n❌ Found {len(missing_templates)} missing templates:")
    display(missing_templates[['title', 'link', 'file_path']])
else:
    print("\n✅ All templates exist!")

## Create Missing Pages

Generate template files for any missing pages identified in the previous step.

In [ ]:
def create_template_file(template_info):
    """Create a template file for a missing link."""
    template_path = Path(template_info['template_path'])
    
    # Create parent directories if they don't exist
    template_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create template content
    title = template_info['title']
    today = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    content = DEFAULT_TEMPLATE.format(title=title, date=today)
    
    try:
        with open(template_path, 'w') as f:
            f.write(content)
        return True
    except Exception as e:
        print(f"Error creating template {template_path}: {str(e)}")
        return False

# Ask for confirmation before creating files
if not missing_templates.empty:
    confirmation = input(f"Create {len(missing_templates)} missing template files? (y/n): ")
    
    if confirmation.lower() == 'y':
        created_files = []
        failed_files = []
        
        for _, template_info in missing_templates.iterrows():
            success = create_template_file(template_info)
            if success:
                created_files.append(template_info['file_path'])
            else:
                failed_files.append(template_info['file_path'])
        
        print(f"\n✅ Created {len(created_files)} template files")
        if failed_files:
            print(f"❌ Failed to create {len(failed_files)} template files")
            print("\nFailed files:")
            for file in failed_files:
                print(f" - {file}")
    else:
        print("Template creation skipped.")
else:
    print("No missing templates to create.")

## Update Sidebar Links

Ensure all sidebar links are correctly formatted and point to valid template files.

In [ ]:
def update_sidebar_links(sidebar_config):
    """
    Update sidebar links to ensure they point to existing files.
    - Adds .html extension if missing
    - Ensures links start with /
    """
    def update_item(item):
        if 'link' in item:
            link = item['link']
            
            # Ensure link starts with /
            if not link.startswith('/'):
                link = '/' + link
            
            # Add .html extension if missing and not ending with /
            if not link.endswith('/') and not link.endswith('.html'):
                link = f"{link}.html"
            
            # Update the link
            item['link'] = link
        
        # Process child items if they exist
        if 'children' in item:
            for child in item['children']:
                update_item(child)
        
        return item
    
    # Create a deep copy of the sidebar config to modify
    updated_config = sidebar_config.copy()
    
    # Update each top-level item in the sidebar
    if 'sidebar' in updated_config:
        updated_config['sidebar'] = [update_item(item) for item in updated_config['sidebar']]
    
    return updated_config

# Check if we need to update the sidebar configuration
if not missing_templates.empty and 'y' in input("Update sidebar links to ensure correct formatting? (y/n): ").lower():
    updated_sidebar_config = update_sidebar_links(sidebar_config)
    
    # Save a backup of the original config
    backup_path = SIDEBAR_CONFIG_PATH.with_suffix('.json.bak')
    shutil.copy2(SIDEBAR_CONFIG_PATH, backup_path)
    print(f"Original sidebar config backed up to {backup_path}")
    
    # Save the updated config
    try:
        with open(SIDEBAR_CONFIG_PATH, 'w') as f:
            json.dump(updated_sidebar_config, f, indent=2)
        print(f"✅ Updated sidebar configuration saved to {SIDEBAR_CONFIG_PATH}")
    except Exception as e:
        print(f"❌ Error saving updated sidebar configuration: {str(e)}")
else:
    print("Sidebar link update skipped.")

## Generate Link Status Report

Create a comprehensive report showing the status of all sidebar links and template files.

In [ ]:
# Re-check the status after creating any missing templates
def recheck_template_status():
    """Re-check the status of all templates after creating missing ones."""
    updated_status = []
    for link_info in sidebar_links:
        link = link_info['link']
        result = check_template_exists(link)
        result['title'] = link_info['title']
        result['parent_path'] = link_info['parent_path']
        updated_status.append(result)
    
    return pd.DataFrame(updated_status)

# Generate the report
updated_status_df = recheck_template_status()
updated_status_df['status'] = updated_status_df['exists'].apply(
    lambda x: '✅ Exists' if x else '❌ Missing'
)

# Generate report summary
report = f"""
# Sidebar Links Status Report
Generated on: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Summary
- Total Links: {len(updated_status_df)}
- Existing Templates: {updated_status_df['exists'].sum()}
- Missing Templates: {len(updated_status_df) - updated_status_df['exists'].sum()}

"""

# Add section for still missing templates
still_missing = updated_status_df[~updated_status_df['exists']]
if not still_missing.empty:
    report += f"""
## Still Missing Templates ({len(still_missing)})
The following templates still need to be created:

| Title | Link Path | File Path |
|-------|-----------|-----------|
"""
    for _, row in still_missing.iterrows():
        report += f"| {row['title']} | {row['link']} | {row['file_path']} |\n"

# Add completed section
report += f"""
## Complete Template Status

| Title | Parent | Link | File Path | Status |
|-------|--------|------|-----------|--------|
"""
for _, row in updated_status_df.iterrows():
    report += f"| {row['title']} | {row['parent_path']} | {row['link']} | {row['file_path']} | {row['status']} |\n"

# Display the report using Markdown
display(Markdown(report))

# Save the report
report_path = BASE_DIR / 'sidebar_link_report.md'
with open(report_path, 'w') as f:
    f.write(report)

print(f"Report saved to {report_path}")

## Conclusion

This notebook helps ensure that all sidebar links in your application correspond to existing template files. It:

1. Identified all sidebar links from the configuration
2. Checked for missing templates in the /templates directory
3. Created placeholder files for any missing templates
4. Updated sidebar links to ensure correct formatting
5. Generated a comprehensive status report

To maintain your application structure:
- Run this notebook whenever you update the sidebar configuration
- Review and enhance any automatically generated template files
- Keep the sidebar links and templates in sync